<a href="https://colab.research.google.com/github/Kirrrk-git/rise-unet-rzsm/blob/mindanao-adaptation/notebooks/13_mindanao_recursive_degradation_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 13: Phase 23 — Mindanao RISE-UNet Recursive Degradation Diagnostic & Post-A0 Gate 2 (G2-R) Decision

**Milestone**: Phase 23 (Recursive Degradation Diagnostic) & Post-A0 Gate 2 (G2-R: Recursive Refinement Decision Gate)  
**Reference Contracts**: [`contracts/A0/A0_Mindanao_Baseline_Contract.yaml`](../contracts/A0/A0_Mindanao_Baseline_Contract.yaml) | [`contracts/A0/PHASE_23_DIAGNOSTIC_CONTRACT.yaml`](../contracts/A0/PHASE_23_DIAGNOSTIC_CONTRACT.yaml)  
**Authority Status**: Pre-Analysis Specification Frozen (`PRE_ANALYSIS_SPECIFICATION_FROZEN`)  
**Hardware Target**: Physical GPU (NVIDIA Tesla T4 / High-RAM) or CPU on Google Colab  
**Evaluation Cohort**: Validation Partition 2022–2023 ($N=194$ usable cases, 126 active evaluation cells)  
**Sealed Test Quarantine**: 2024–2025 Holdout ($N=202$ usable cases) strictly untouched until Phase 26.  


## 1. Environment Setup, Dependencies & Google Cloud Storage Lake Synchronization

In [1]:
# Install dependencies, mount Google Drive/GCS, and configure repository path
import os
import sys
import subprocess
from pathlib import Path

# Install necessary geospatial and analytical dependencies if running in Colab
if 'google.colab' in sys.modules:
    !pip install -q keras-cv xarray netcdf4 pyyaml pandas numpy matplotlib scipy

# Self-healing: clear any dummy or stale keras_cv imports from memory
if 'keras_cv' in sys.modules and not hasattr(sys.modules['keras_cv'], 'layers'):
    del sys.modules['keras_cv']
if 'parent_study_ex29.function.modelRzsmRelu' in sys.modules:
    del sys.modules['parent_study_ex29.function.modelRzsmRelu']

REPO_DIR = Path('/content/rise-unet-rzsm') if Path('/content').exists() else Path('.').resolve()
if not REPO_DIR.exists():
    !git clone -b mindanao-adaptation https://github.com/Kirrrk-git/rise-unet-rzsm.git /content/rise-unet-rzsm
    REPO_DIR = Path('/content/rise-unet-rzsm')
else:
    !cd /content/rise-unet-rzsm && git pull origin mindanao-adaptation

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Working directory: {REPO_DIR}')

# Authenticate Google Cloud Storage if running in Colab
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print('GCS Authentication complete.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 63.5 MB/s eta 0:00:00
Cloning into '/content/rise-unet-rzsm'...
remote: Enumerating objects: 1518, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 1518 (delta 80), reused 87 (delta 41), pack-reused 1372 (from 1)
Receiving objects: 100% (1518/1518), 248.05 MiB | 18.58 MiB/s, done.
Resolving deltas: 100% (854/854), done.
Updating files: 100% (408/408), done.
Working directory: /content/rise-unet-rzsm
GCS Authentication complete.


## 2. Pre-Analysis Contract & Authoritative Evaluation Domain Verification

In [2]:
import yaml
import xarray as xr
import numpy as np
import pandas as pd

# 1. Load frozen diagnostic contract
contract_path = REPO_DIR / "contracts" / "A0" / "PHASE_23_DIAGNOSTIC_CONTRACT.yaml"
with open(contract_path, "r", encoding="utf-8") as f:
    diagnostic_contract = yaml.safe_load(f)

print("=" * 85)
print(f"CONTRACT TITLE    : {diagnostic_contract['contract_title']}")
print(f"CONTRACT VERSION  : {diagnostic_contract['contract_version']}")
print(f"GOVERNANCE STATUS : {diagnostic_contract['governance_status']}")
print(f"USABLE VAL COHORT : {diagnostic_contract['evaluation_cohort']['usable_validation_denominator']} cases")
print(f"ACTIVE EVAL CELLS : {diagnostic_contract['evaluation_cohort']['spatial_domain']['active_evaluation_cells']} cells")
print(f"SEALED TEST STATUS: {diagnostic_contract['evaluation_cohort']['sealed_test_quarantine']['status']}")
print("=" * 85)

# 2. Load and verify 126-cell binary evaluation mask
mask_path = REPO_DIR / "processed" / "grid" / "mindanao_eval_mask_025.nc"
with xr.open_dataset(mask_path) as ds:
    var_name = "eval_mask" if "eval_mask" in ds else list(ds.data_vars.keys())[0]
    eval_mask = ds[var_name].values.astype(bool)

active_count = int(np.sum(eval_mask))
assert active_count == 126, f"Expected 126 active cells, got {active_count}"
print(f"[PASS] Evaluation mask verified: shape={eval_mask.shape}, active_cells={active_count}, ocean_cells={eval_mask.size - active_count}")

# 3. Load validation manifest
val_manifest_path = REPO_DIR / "manifests" / "splits" / "val_cases.csv"
df_val = pd.read_csv(val_manifest_path)
print(f"[PASS] Validation manifest loaded: {len(df_val)} cycles")

CONTRACT TITLE    : Mindanao RISE-UNet Phase 23 Pre-Analysis Specification & Diagnostic Contract
CONTRACT VERSION  : v1.0.0-pre-analysis-freeze
GOVERNANCE STATUS : PRE_ANALYSIS_SPECIFICATION_FROZEN
USABLE VAL COHORT : 194 cases
ACTIVE EVAL CELLS : 126 cells
SEALED TEST STATUS: STRICTLY_SEALED_UNTIL_PHASE_26
[PASS] Evaluation mask verified: shape=(32, 48), active_cells=126, ocean_cells=1410
[PASS] Validation manifest loaded: 210 cycles


## 3. Model A0 Production Checkpoints Restoration & Verification

Verifies and restores all 12 frozen Model A0 checkpoints (Seeds 42, 123, 456 across Leads W1 to W4) from the GCS lake (`gs://rise-unet-rzsm/checkpoints/A0/`).

In [3]:
import sys
if 'keras_cv' in sys.modules and not hasattr(sys.modules['keras_cv'], 'layers'):
    del sys.modules['keras_cv']
if 'parent_study_ex29.function.modelRzsmRelu' in sys.modules:
    del sys.modules['parent_study_ex29.function.modelRzsmRelu']

from src.models.a0_unet import build_a0_unet, ensure_keras_compatibility
ensure_keras_compatibility()

SEEDS = [42, 123, 456]
LEADS = [1, 2, 3, 4]
CHECKPOINTS_DIR = REPO_DIR / 'checkpoints' / 'A0'

# Sync checkpoints from GCS if running in Colab and local checkpoints are absent
for seed in SEEDS:
    for lead in LEADS:
        ckpt_path = CHECKPOINTS_DIR / f'seed_{seed}' / f'lead_{lead}' / 'best_model.weights.h5'
        if not ckpt_path.is_file():
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            gcs_uri = f'gs://rise-unet-rzsm/checkpoints/A0/seed_{seed}/lead_{lead}/best_model.weights.h5'
            print(f'Downloading checkpoint from {gcs_uri}...')
            !gsutil -q cp {gcs_uri} {ckpt_path}
        assert ckpt_path.is_file(), f'Missing checkpoint: {ckpt_path}'

print('[PASS] All 12 Model A0 production checkpoints verified present.')


[PASS] All 12 Model A0 production checkpoints verified present.


## 4. Multi-Protocol Validation Inference Engine (Protocols 1, 2, and 4)

Executes parallel inference across all 194 validation cases under:
- **Protocol 1**: Standard Autoregressive Recursion ($W_1 \to \hat{y}_{W1} \to W_2 \to \dots$)
- **Protocol 2**: Oracle Counterfactual Replacement ($y_{W1}^{\text{true}}, y_{W2}^{\text{true}}, y_{W3}^{\text{true}}$)
- **Protocol 4**: Controlled Error-Injection ($\hat{y}_{W1} \pm \epsilon$ with $[0, 1]$ clipping rule)

In [4]:
from src.data.tf_dataset import prepare_case_lead_tensors
from src.evaluation.recursive_diagnostic import (
    assemble_oracle_priors,
    apply_boundary_clipped_perturbation,
    compute_case_level_metrics,
    compute_paired_differences,
    moving_block_bootstrap,
    wilcoxon_paired_test,
    holm_bonferroni_correction,
    compute_cohens_d_z,
    evaluate_gate2_rules,
)

CASES_DIR = REPO_DIR / "processed" / "cases" / "production"
if not CASES_DIR.is_dir():
    # Sync validation cases from GCS
    CASES_DIR.mkdir(parents=True, exist_ok=True)
    print("Syncing validation cases from gs://rise-unet-rzsm/processed/cases/production/...")
    !gsutil -m -q cp "gs://rise-unet-rzsm/processed/cases/production/CASE_2022*.npz" {CASES_DIR}/
    !gsutil -m -q cp "gs://rise-unet-rzsm/processed/cases/production/CASE_2023*.npz" {CASES_DIR}/

val_case_files = sorted(list(CASES_DIR.glob("CASE_*.npz")))
print(f"Found {len(val_case_files)} validation case NPZ files.")

# Pre-load case data dictionary
loaded_cases = {}
for cp in val_case_files:
    cid = cp.stem.replace("CASE_", "")
    with np.load(cp) as d:
        loaded_cases[cid] = {k: d[k] for k in d.files}

print(f"Successfully pre-loaded {len(loaded_cases)} validation cases in memory.")

Syncing validation cases from gs://rise-unet-rzsm/processed/cases/production/...
Found 194 validation case NPZ files.
Successfully pre-loaded 194 validation cases in memory.


## 5. Execution of Diagnostic Protocols Across 3 Seeds & 4 Leads

Runs model predictions across all 3 independent training replicates (Seeds 42, 123, 456) under Protocols 1, 2, and 4.

In [5]:
PERTURBATIONS = [-0.50, -0.25, -0.10, -0.05, 0.05, 0.10, 0.25, 0.50]

diagnostic_records = {
    'seeds': SEEDS,
    'leads': LEADS,
    'by_seed': {},
    'lead_summary': {},
    'clipping_census': {},
    'divergence_by_lead_eps': {str(l): {} for l in [2, 3, 4]},
}

# Execute evaluation across seeds
for seed in SEEDS:
    print(f'\n>>> EVALUATING MODEL A0 TRAINING REPLICATE SEED {seed} <<<')
    diagnostic_records['by_seed'][seed] = {'p1_rec': {}, 'p2_ora': {}, 'p4_pert': {}}

    # Load models for all 4 leads
    models = {}
    for lead in LEADS:
        m = build_a0_unet(lead=lead)
        ckpt = CHECKPOINTS_DIR / f'seed_{seed}' / f'lead_{lead}' / 'best_model.weights.h5'
        m.load_weights(str(ckpt))
        models[lead] = m

    # Protocol 1: Standard Recursion
    # Protocol 2: Oracle Counterfactual
    rec_preds = {l: {} for l in LEADS}
    ora_preds = {l: {} for l in LEADS}
    p1_metrics = {l: [] for l in LEADS}
    p2_metrics = {l: [] for l in LEADS}

    for cid, cdata in loaded_cases.items():
        # Lead 1 (same for both)
        x1, y1 = prepare_case_lead_tensors(cdata, lead=1)
        p1_raw = models[1](x1, training=False)[2].numpy()
        rec_preds[1][cid] = p1_raw
        ora_preds[1][cid] = p1_raw
        m1 = compute_case_level_metrics(cdata['y_w1'], p1_raw, eval_mask)
        p1_metrics[1].append(m1)
        p2_metrics[1].append(m1)

        # Leads 2, 3, 4
        for lead in (2, 3, 4):
            # Protocol 1: Ingest prior predictions
            prior_rec = {pl: rec_preds[pl][cid] for pl in range(1, lead)}
            x_rec, _ = prepare_case_lead_tensors(cdata, lead=lead, y_hat_prev=prior_rec)
            p_rec = models[lead](x_rec, training=False)[2].numpy()
            rec_preds[lead][cid] = p_rec
            m_rec = compute_case_level_metrics(cdata[f'y_w{lead}'], p_rec, eval_mask)
            p1_metrics[lead].append(m_rec)

            # Protocol 2: Ingest Oracle ground truth
            prior_ora = assemble_oracle_priors(cdata, lead=lead)
            x_ora, _ = prepare_case_lead_tensors(cdata, lead=lead, y_hat_prev=prior_ora)
            p_ora = models[lead](x_ora, training=False)[2].numpy()
            ora_preds[lead][cid] = p_ora
            m_ora = compute_case_level_metrics(cdata[f'y_w{lead}'], p_ora, eval_mask)
            p2_metrics[lead].append(m_ora)

    diagnostic_records['by_seed'][seed]['p1_rec'] = p1_metrics
    diagnostic_records['by_seed'][seed]['p2_ora'] = p2_metrics
    print(f'Completed Protocol 1 and Protocol 2 evaluation for Seed {seed}.')



>>> EVALUATING MODEL A0 TRAINING REPLICATE SEED 42 <<<
Completed Protocol 1 and Protocol 2 evaluation for Seed 42.

>>> EVALUATING MODEL A0 TRAINING REPLICATE SEED 123 <<<
Completed Protocol 1 and Protocol 2 evaluation for Seed 123.

>>> EVALUATING MODEL A0 TRAINING REPLICATE SEED 456 <<<
Completed Protocol 1 and Protocol 2 evaluation for Seed 456.


## 6. Statistical Significance & Moving-Block Bootstrap CIs Across Replicate Seeds

In [6]:
# Compute multi-seed summary, primary Moving-Block Bootstrap (MBB), and supplementary Wilcoxon test
lead_summary = {}
raw_p_boot_values = {}
supp_wilcoxon_values = {}

print("=" * 115)
print("LEAD | PROTOCOL 1 (REC MAE) | PROTOCOL 2 (ORA MAE) | RECURSIVE GAP ΔMAE (95% MBB CI) | COHEN'S dz | PRIMARY p_boot | SUPP WILCOXON p")
print("-" * 115)

for lead in [1, 2, 3, 4]:
    rec_mae_seeds = []
    ora_mae_seeds = []
    gaps_seeds = []

    all_seed_diffs = []
    for seed in SEEDS:
        s_rec = [m['mae'] for m in diagnostic_records["by_seed"][seed]["p1_rec"][lead]]
        s_ora = [m['mae'] for m in diagnostic_records["by_seed"][seed]["p2_ora"][lead]]
        rec_mae_seeds.append(np.mean(s_rec))
        ora_mae_seeds.append(np.mean(s_ora))
        diff = compute_paired_differences(s_rec, s_ora)
        all_seed_diffs.append(diff)

    mean_diff_cases = np.mean(all_seed_diffs, axis=0)  # Average across seeds per case
    # Primary inferential procedure: Circular Moving-Block Bootstrap (L=4 cycles ~ 2 months)
    mean_gap, ci_lo, ci_hi, p_boot = moving_block_bootstrap(mean_diff_cases, block_length=4, n_boot=10000, seed=42)
    # Supplementary nonparametric paired check
    stat_w, pval_w = wilcoxon_paired_test(mean_diff_cases)
    dz = compute_cohens_d_z(mean_diff_cases)

    if lead > 1:
        raw_p_boot_values[lead] = p_boot
        supp_wilcoxon_values[lead] = pval_w

    rec_m = np.mean(rec_mae_seeds)
    ora_m = np.mean(ora_mae_seeds)
    rel_gap = (mean_gap / rec_m) * 100.0 if rec_m > 0 else 0.0

    lead_summary[lead] = {
        "rec_mae_mean": float(rec_m),
        "rec_mae_sd": float(np.std(rec_mae_seeds)),
        "ora_mae_mean": float(ora_m),
        "ora_mae_sd": float(np.std(ora_mae_seeds)),
        "delta_mae_mean": float(mean_gap),
        "delta_mae_ci_lower": float(ci_lo),
        "delta_mae_ci_upper": float(ci_hi),
        "relative_gap_pct": float(rel_gap),
        "cohens_d_z": float(dz),
        "p_boot": float(p_boot),
        "raw_p": float(p_boot),  # Primary inference for Holm-Bonferroni
        "supp_wilcoxon_p": float(pval_w),
    }

    gap_str = f"{mean_gap:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}]"
    print(f" W{lead}  | {rec_m:.4f} ± {np.std(rec_mae_seeds):.4f}     | {ora_m:.4f} ± {np.std(ora_mae_seeds):.4f}     | {gap_str:30s} | {dz:+.3f}    | {p_boot:.4e}     | {pval_w:.4e}")

print("=" * 115)

# Step-down Holm-Bonferroni correction across Leads W2, W3, W4 on primary bootstrap p-values
hb_results = holm_bonferroni_correction(raw_p_boot_values, alpha=0.05)
for l, res in hb_results.items():
    lead_summary[l]["adj_p"] = res["adj_p"]
    lead_summary[l]["rejected"] = res["rejected"]
    print(f"Lead W{l} Step-Down Holm-Bonferroni: Raw p_boot = {res['raw_p']:.4e} -> Adjusted p_boot = {res['adj_p']:.4e} | H0 Rejected: {res['rejected']}")


LEAD | PROTOCOL 1 (REC MAE) | PROTOCOL 2 (ORA MAE) | RECURSIVE GAP ΔMAE (95% MBB CI) | COHEN'S dz | PRIMARY p_boot | SUPP WILCOXON p
-------------------------------------------------------------------------------------------------------------------
 W1  | 0.0340 ± 0.0007     | 0.0340 ± 0.0007     | +0.0000 [+0.0000, +0.0000]     | +0.000    | 1.0000e+00     | 1.0000e+00
 W2  | 0.0526 ± 0.0016     | 0.0450 ± 0.0031     | +0.0076 [+0.0059, +0.0098]     | +0.642    | 0.0000e+00     | 4.7087e-23
 W3  | 0.0719 ± 0.0168     | 0.0465 ± 0.0031     | +0.0254 [+0.0216, +0.0301]     | +1.201    | 0.0000e+00     | 2.4106e-29
 W4  | 0.0898 ± 0.0288     | 0.0502 ± 0.0021     | +0.0396 [+0.0328, +0.0470]     | +1.269    | 0.0000e+00     | 1.5609e-28
Lead W2 Step-Down Holm-Bonferroni: Raw p_boot = 0.0000e+00 -> Adjusted p_boot = 0.0000e+00 | H0 Rejected: True
Lead W3 Step-Down Holm-Bonferroni: Raw p_boot = 0.0000e+00 -> Adjusted p_boot = 0.0000e+00 | H0 Rejected: True
Lead W4 Step-Down Holm-Bonferroni

## 7. Protocol 4: Controlled Error-Injection Sensitivity & Boundary Saturation Census

Evaluates downstream divergence $D_k(\epsilon)$ under deterministic member-wise perturbation:
$$\hat{y}_{W1,m}' = \operatorname{clip}(\hat{y}_{W1,m} + \epsilon, 0.0, 1.0) \quad \text{for every } m \in \{1, \dots, 11\}$$
This preserves ensemble member correspondence; ensemble spread is unchanged on unclipped cells, while any boundary-induced spread changes are explicitly recorded by the clipping census.

In [7]:
# Evaluate Protocol 4 across perturbation levels
print("=" * 85)
print("PERTURBATION OFFSET ε | EVALUATIONS CLIPPED TO [0, 1] (%) | DOWNSTREAM DIVERGENCE D_k(ε) [RMSE]")
print("                      |                                   |  W2        W3        W4")
print("-" * 85)

clipping_census = {}
divergence_summary = {str(l): {} for l in [2, 3, 4]}

# Use Seed 42 for representative perturbation tracing
m1 = build_a0_unet(lead=1); m1.load_weights(str(CHECKPOINTS_DIR / 'seed_42' / 'lead_1' / 'best_model.weights.h5'))
m2 = build_a0_unet(lead=2); m2.load_weights(str(CHECKPOINTS_DIR / 'seed_42' / 'lead_2' / 'best_model.weights.h5'))
m3 = build_a0_unet(lead=3); m3.load_weights(str(CHECKPOINTS_DIR / 'seed_42' / 'lead_3' / 'best_model.weights.h5'))
m4 = build_a0_unet(lead=4); m4.load_weights(str(CHECKPOINTS_DIR / 'seed_42' / 'lead_4' / 'best_model.weights.h5'))

for eps in PERTURBATIONS:
    clipped_counts = []
    div_w2 = []
    div_w3 = []
    div_w4 = []

    for cid, cdata in loaded_cases.items():
        x1, _ = prepare_case_lead_tensors(cdata, lead=1)
        p1_base = m1(x1, training=False)[2].numpy()

        # Apply perturbation with clipping
        p1_pert, c_cnt, c_pct = apply_boundary_clipped_perturbation(p1_base, epsilon=eps, eval_mask=eval_mask)
        clipped_counts.append(c_pct)

        # Unperturbed cascade baseline
        p2_base = m2(prepare_case_lead_tensors(cdata, lead=2, y_hat_prev={1: p1_base})[0], training=False)[2].numpy()
        p3_base = m3(prepare_case_lead_tensors(cdata, lead=3, y_hat_prev={1: p1_base, 2: p2_base})[0], training=False)[2].numpy()
        p4_base = m4(prepare_case_lead_tensors(cdata, lead=4, y_hat_prev={1: p1_base, 2: p2_base, 3: p3_base})[0], training=False)[2].numpy()

        # Perturbed cascade downstream
        p2_pert = m2(prepare_case_lead_tensors(cdata, lead=2, y_hat_prev={1: p1_pert})[0], training=False)[2].numpy()
        p3_pert = m3(prepare_case_lead_tensors(cdata, lead=3, y_hat_prev={1: p1_pert, 2: p2_pert})[0], training=False)[2].numpy()
        p4_pert = m4(prepare_case_lead_tensors(cdata, lead=4, y_hat_prev={1: p1_pert, 2: p2_pert, 3: p3_pert})[0], training=False)[2].numpy()

        # Compute divergence over 126 active cells
        div_w2.append(np.sqrt(np.mean((p2_pert[:, eval_mask] - p2_base[:, eval_mask]) ** 2)))
        div_w3.append(np.sqrt(np.mean((p3_pert[:, eval_mask] - p3_base[:, eval_mask]) ** 2)))
        div_w4.append(np.sqrt(np.mean((p4_pert[:, eval_mask] - p4_base[:, eval_mask]) ** 2)))

    mean_clip = float(np.mean(clipped_counts))
    clipping_census[str(eps)] = mean_clip
    divergence_summary["2"][str(eps)] = float(np.mean(div_w2))
    divergence_summary["3"][str(eps)] = float(np.mean(div_w3))
    divergence_summary["4"][str(eps)] = float(np.mean(div_w4))

    print(f"  ε = {eps:+0.2f}               | {mean_clip:6.2f}%                           | {np.mean(div_w2):.4f}    {np.mean(div_w3):.4f}    {np.mean(div_w4):.4f}")

print("=" * 85)
diagnostic_records["clipping_census"] = clipping_census
diagnostic_records["divergence_by_lead_eps"] = divergence_summary
diagnostic_records["lead_summary"] = lead_summary

PERTURBATION OFFSET ε | EVALUATIONS CLIPPED TO [0, 1] (%) | DOWNSTREAM DIVERGENCE D_k(ε) [RMSE]
                      |                                   |  W2        W3        W4
-------------------------------------------------------------------------------------
  ε = -0.50               |  16.52%                           | 0.2890    0.2979    0.1566
  ε = -0.25               |   0.00%                           | 0.1364    0.1819    0.0961
  ε = -0.10               |   0.00%                           | 0.0498    0.0765    0.0395
  ε = -0.05               |   0.00%                           | 0.0242    0.0373    0.0192
  ε = +0.05               |   0.08%                           | 0.0241    0.0344    0.0274
  ε = +0.10               |   0.27%                           | 0.0498    0.0699    0.0526
  ε = +0.25               |   6.42%                           | 0.1353    0.2096    0.2119
  ε = +0.50               |  83.48%                           | 0.1905    0.3942    0.3613


## 8. Post-A0 Gate 2 (G2-R) Formal Decision Evaluation & Decision Dossier Generation

Evaluates all empirical evidence against the pre-registered GO / NO-GO criteria in `contracts/A0/PHASE_23_DIAGNOSTIC_CONTRACT.yaml`.
Formally disambiguated from Pre-Production Gate 2 (Hardware Profiling & VRAM Feasibility).


In [9]:
import json
import time

from src.evaluation.recursive_diagnostic import evaluate_gate2_rules

# Evaluate Post-A0 Gate 2 (G2-R) decision
# Check replicate consistency: Delta E_k > 0 across all 3 seeds in downstream leads W2..W4
seed_consistent = all(
    all(
        np.mean([m['mae'] for m in diagnostic_records['by_seed'][s]['p1_rec'][l]]) >
        np.mean([m['mae'] for m in diagnostic_records['by_seed'][s]['p2_ora'][l]])
        for l in [2, 3, 4]
    )
    for s in SEEDS
)

gate2_decision = evaluate_gate2_rules(
    lead_summary=lead_summary,
    seed_consistency=seed_consistent,
    divergence_monotonic=divergence_summary,
)

print('=' * 80)
print(f"POST-A0 GATE 2 (G2-R) VERDICT: {gate2_decision['verdict']}")
print(f"ACTION: {gate2_decision['action']}")
print('=' * 80)
print(json.dumps(gate2_decision['criteria'], indent=2))

# Save telemetry records and Gate 2 decision
out_json_path = REPO_DIR / 'logs' / 'phase23_recursive_diagnostic_results.json'
out_json_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'validation_cases_count': len(loaded_cases),
        'evaluation_cells_count': 126,
        'seeds': SEEDS,
        'perturbations_evaluated': PERTURBATIONS,
        'clipping_census': clipping_census,
        'divergence_by_lead_eps': divergence_summary,
        'lead_summary': lead_summary,
        'gate2_decision': gate2_decision,
    }, f, indent=2)
print(f'Saved diagnostic telemetry to: {out_json_path}')


POST-A0 GATE 2 (G2-R) VERDICT: GO
ACTION: Proceed to Phase 24 (Model A1 Lead-Aware Recursive Residual Refinement)
{
  "GO-1_statistical_significance": {
    "passed": true,
    "w3_rejected": true,
    "w4_rejected": true
  },
  "GO-2_practical_effect_size": {
    "passed": true,
    "w3_cohens_d_z": 1.2013971358745263,
    "w4_cohens_d_z": 1.2691589565531978,
    "w3_relative_gap_pct": 35.37523645771402,
    "w4_relative_gap_pct": 44.116260998595294
  },
  "GO-3_replicate_consistency": {
    "passed": true
  },
  "GO-4_perturbation_sensitivity": {
    "passed": true,
    "details": {
      "lead_3": {
        "d_pos_05": 0.03439375385642052,
        "d_pos_10": 0.06993638724088669,
        "positive_monotonic": true,
        "d_neg_05": 0.03726876154541969,
        "d_neg_10": 0.07648175209760666,
        "negative_monotonic": true,
        "passed": true
      },
      "lead_4": {
        "d_pos_05": 0.02738095261156559,
        "d_pos_10": 0.05256462097167969,
        "positive_mono

## 9. Publication Figures & Cloud Lake Dual Synchronization

In [10]:
from src.evaluation.recursive_diagnostic import generate_diagnostic_plots

plot_path = REPO_DIR / 'figures' / 'phase23_recursive_degradation_composite.png'
generate_diagnostic_plots(diagnostic_records, plot_path)

# Dual Lake Synchronization to gs://rise-unet-rzsm/
print('Synchronizing Phase 23 diagnostic artifacts to GCS lake...')
!gsutil cp {out_json_path} gs://rise-unet-rzsm/logs/
!gsutil cp {plot_path} gs://rise-unet-rzsm/figures/
print('[PASS] Dual artifact synchronization complete.')


Synchronizing Phase 23 diagnostic artifacts to GCS lake...
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/rise-unet-rzsm/logs/phase23_recursive_diagnostic_results.json [Content-Type=application/json]...
\
Operation completed over 1 objects/5.0 KiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/rise-unet-rzsm/figures/phase23_recursive_degradation_composite.png [Content-Type=image/png]...
- [1 files][567.3 KiB/567.3 KiB]                                                
Operation complete

---
## Executive Findings & Scientific Certification Summary

### Notebook 13: Phase 23 Recursive Degradation Diagnostic & Post-A0 Gate 2 (G2-R) Decision

| Diagnostic Component | Pre-Registered Standard & Contract Target | Methodological Specification & Measurement | Scientific Status |
| :--- | :--- | :--- | :---: |
| **Evaluation Cohort** | 2022–2023 Validation Partition | 194 usable cases (excluding 16 audited provider MARS exceptions) | `[PRE-ANALYSIS FROZEN]` |
| **Active Land Domain** | 126 evaluation land cells | Candidate A regular $0.25^\circ$ mesh; 1,410 ocean cells zero-filled | `[FROZEN]` |
| **Sealed Holdout Protection** | 2024–2025 Sealed Test Set ($N=202$) | Strictly quarantined; zero access permitted during Phase 23 | `[STRICTLY QUARANTINED]` |
| **Protocol 1 (Recursive)** | Standard autoregressive unrolling | Models ingest prior predicted states along recursive chain ($W_1 \to W_4$) | `[STANDARD BASE]` |
| **Protocol 2 (Oracle)** | Ground-truth counterfactual replacement | Ingests verifying truth $y^{\text{true}}$ holding all issue dates and weights invariant | `[PRIMARY DIAGNOSTIC]` |
| **Primary Error Gap** | $\Delta E_k = E_k^{\text{rec}} - E_k^{\text{ora}}$ | Quantifies recursive error compounding beyond residual forecast error | `[FROZEN FORMULATION]` |
| **Protocol 4 (Sensitivity)** | Member-wise offset $\hat{y}' = \operatorname{clip}(\hat{y} + \epsilon, 0, 1)$ | Deterministic application preserves member correspondence and unclipped spread | `[FROZEN PROTOCOL]` |
| **Clipping Census** | Land boundary saturation tracking | Explicit active evaluation cell clipping percentages recorded per perturbation | `[CENSUS AUDIT]` |
| **Monotonicity Standard** | Downstream divergence $D_k(\epsilon)$ across magnitude $|\epsilon|$ | Evaluated separately by sign ($+0.05 \to +0.10$ and $-0.05 \to -0.10$); $\pm 0.25, \pm 0.50$ stress tests | `[MAGNITUDE MONOTONIC]` |
| **Primary Inference** | Circular Moving-Block Bootstrap (MBB, $L=4, B=10000$) | Uncentered 95% CIs + centered null series $d_i^{(0)} = d_i - \bar{d}$ for calibrated $p_{\text{boot}}$ | `[PRIMARY INFERENCE]` |
| **Supplementary Check** | Paired Wilcoxon signed-rank test | Two-sided non-parametric test reported alongside primary block bootstrap | `[SUPPLEMENTARY]` |
| **Multiple Testing** | Step-down Holm-Bonferroni correction | Family-wise error rate controlled at $\alpha=0.05$ across Leads W2, W3, W4 | `[CONTROLLED]` |
| **Decision Gate Title** | Post-A0 Gate 2 (G2-R) | Formally disambiguated from Pre-Production Gate 2 (Hardware Profiling) | `[FORMAL GATE]` |
| **Decision Criteria** | GO-1 ($p_{\text{boot}} < 0.05$), GO-2 ($d_z \ge 0.20$), GO-3 (Seeds), GO-4 ($|\epsilon|$) | Locked prior to inspecting results; dictates Phase 24 vs Model A0 freeze | `[FROZEN CRITERIA]` |
| **Reference Contract** | Phase 23 Diagnostic Contract | [`contracts/A0/PHASE_23_DIAGNOSTIC_CONTRACT.yaml`](../contracts/A0/PHASE_23_DIAGNOSTIC_CONTRACT.yaml) | `[PRE-ANALYSIS FROZEN]` |

**Key Takeaways for Gate 2 Decision**:
1. The diagnostic protocol design, mathematical definitions, and decision thresholds are frozen before analyzing validation results, preventing post-hoc justification.
2. The empirical evidence generated by this notebook determines whether to proceed to Phase 24 (Model A1 Lead-Aware Residual Refinement) or formally halt neural modifications and freeze Model A0 as the regional benchmark.
3. Full parity between local artifacts and `gs://rise-unet-rzsm/` ensures complete reproducibility across all platforms.